#IMPORT LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')


#Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/DataSets/spam.csv', encoding='latin-1')[['v1', 'v2']]
df.columns = ['label', 'message']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Encode label


In [ ]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})


#Test and Train Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['message'], df['label'], test_size=0.2, random_state=42)


#Vectorization Technique

In [ ]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


#Model

In [ ]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)


MultinomialNB()

#Prediction

In [ ]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
test_sms = input("\n Enter an SMS message to check if it's SPAM or HAM: ")

test_sms_vector = vectorizer.transform([test_sms])
prediction = model.predict(test_sms_vector)[0]

print(f"\n Message: {test_sms}")
print(f" Prediction: {'SPAM' if prediction == 1 else 'HAM'}")


Accuracy: 0.9623318385650225
[[965   0]
 [ 42 108]]
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       965
           1       1.00      0.72      0.84       150

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115


 Enter an SMS message to check if it's SPAM or HAM: congratulations! you are awarded $5000

 Message: congratulations! you are awarded $5000
 Prediction: SPAM


### Word Embeddings for Spam Detection

In [ ]:
import gensim.downloader as api
import numpy as np

# Load a pre-trained GloVe model

try:
    glove_model = api.load("glove-wiki-gigaword-300")
except Exception as e:
    print(f"Error loading GloVe model: {e}")
    print("Please ensure you have a stable internet connection.")
    glove_model = None

#  to get the vector for a message by averaging word vectors
def message_to_vec(message, model):
    if model is None:
        return np.zeros(300)

    words = message.lower().split()
    word_vectors = [model[word] for word in words if word in model]
    if not word_vectors:
        return np.zeros(model.vector_size)
    return np.mean(word_vectors, axis=0)

if glove_model:
    # Apply the function to create message vectors
    X_train_emb = np.array([message_to_vec(msg, glove_model) for msg in X_train])
    X_test_emb = np.array([message_to_vec(msg, glove_model) for msg in X_test])

    from sklearn.svm import SVC

    embedding_model = SVC(kernel='linear')
    embedding_model.fit(X_train_emb, y_train)

    # Make predictions and evaluate
    y_pred_emb = embedding_model.predict(X_test_emb)

    print("Accuracy (with Embeddings):", accuracy_score(y_test, y_pred_emb))
    print("Confusion Matrix (with Embeddings):\n", confusion_matrix(y_test, y_pred_emb))
    print("Classification Report (with Embeddings):\n", classification_report(y_test, y_pred_emb))


    print("\n SMS Spam Detection ")
    user_sms = input("Enter an SMS message to classify: ")

    vec = message_to_vec(user_sms, glove_model).reshape(1, -1)
    prediction = embedding_model.predict(vec)[0]
    label = "SPAM" if prediction == 1 else "HAM"
    print(f"\nMessage: {user_sms}\nPrediction: {label}")

else:
    print("⚠️ GloVe model not available. Skipping model training and prediction.")

Accuracy (with Embeddings): 0.9426008968609866
Confusion Matrix (with Embeddings):
 [[938  27]
 [ 37 113]]
Classification Report (with Embeddings):
               precision    recall  f1-score   support

           0       0.96      0.97      0.97       965
           1       0.81      0.75      0.78       150

    accuracy                           0.94      1115
   macro avg       0.88      0.86      0.87      1115
weighted avg       0.94      0.94      0.94      1115


🔍 SMS Spam Detection (User Input)
Enter an SMS message to classify: CONGRATULATION ! YOU WON THE $400

📩 Message: CONGRATULATION ! YOU WON THE $400
🔎 Prediction: SPAM


### Test with User Input (TF-IDF Model)